# Machine sound: can a model that has only heard a healthy machine tell when it goes wrong?

**AI anomaly detection for industrial pumps, fans, valves and slide rails, from a microphone.**

A pump rarely fails, so there are many hours of healthy sound and almost no recordings of a fault. The model therefore learns the
healthy sound only, and gives every 10-second clip an *anomaly score*: how far it is from what the model knows as healthy.

This notebook does not train anything. It reads **the scores our system gave to every test clip** and recomputes the results of
three public tests, step by step, in a few seconds and without a GPU.

| | Test | What is new for the model | Our result | Best result of the challenge |
|---|---|---|---|---|
| 1 | DCASE 2020 Task 2 | nothing: known machines | 97.47 % mean AUC | 94.69 % |
| 2 | DCASE 2022 Task 2 | the operating conditions | 71.50 % official score | 70.97 % |
| 3 | DCASE 2025 Task 2 | the machine types | 63.27 % official score | 61.63 % |

> **These are not challenge entries.** The three challenges are closed and their labels are public. Our system was built in 2026 on a
> pre-trained audio model that the 2020 and 2022 teams did not have. The limits are at the end of the notebook.

The training notebooks, the rules written before each run and the full records are in the project's repository:
https://github.com/EngMtaleb/chemai/tree/main/projects/p04_machine_sound

## 0 · Setup

The dataset attached to this notebook holds our own outputs only: one anomaly score per test clip, and the summary files of our runs.
It holds no recording and no label.

In [ ]:
import csv, glob, hashlib, io, json, os, re, sys, urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score


def find_data():
    """The attached dataset: the folder that holds summary.json and the three test folders (unpacked first, if it arrived as a zip)."""
    root = Path(os.environ.get("P04_INPUT", "/kaggle/input"))
    work = Path(os.environ.get("P04_WORK", "/kaggle/working" if Path("/kaggle/working").is_dir() else "."))
    for where in (root, work / "dataset"):
        hits = [p.parent for p in sorted(where.rglob("summary.json")) if (p.parent / "test1_dcase2020").is_dir()]
        if hits:
            return hits[0], work
        for z in sorted(root.rglob("*.zip")):
            with zipfile.ZipFile(z) as zf:
                if any(n.endswith("test1_dcase2020/recorded_official_evaluator_result.csv") for n in zf.namelist()):
                    zf.extractall(work / "dataset")
    raise SystemExit("The dataset is not attached. Use 'Add Input' and add the machine-sound scores dataset.")


DATA, WORK = find_data()
WORK = WORK / "labels"
WORK.mkdir(parents=True, exist_ok=True)
SUMMARY = json.loads((DATA / "summary.json").read_text())      # the numbers recorded by our runs
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#898781"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e1e0d9",
                     "axes.axisbelow": True, "font.size": 11})
print("dataset:", DATA)
print(sum(1 for f in DATA.rglob("*") if f.is_file()), "files")

## 1 · The labels: which test clips were really faulty?

To score a system we need the truth about every test clip. The labels belong to the challenge organisers, so they are **not** in our
dataset. The next cell downloads them from their publishers and checks each file against the fingerprint our own runs recorded.

- **2020:** `eval_data_list.csv`, Zenodo record 3951620.
- **2022:** the official evaluator's repository, at one fixed commit (MIT licence).
- **2025:** the official evaluator's repository, at one fixed commit. Its files come under the organisers' own licence, for
  non-commercial evaluation; this notebook downloads them each time it runs and does not pass them on.

This needs **Internet: on** in the notebook's settings. Without it the notebook still runs, and shows the numbers our runs recorded.

In [ ]:
def fetch(url, name):
    """One file from its publisher (kept in WORK so a second run does not download again). Returns its bytes, or None."""
    p = WORK / name
    if not p.exists():
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r:
                p.write_bytes(r.read())
        except Exception as e:
            print(f"   could not download {name} ({type(e).__name__})")
            return None
    return p.read_bytes()


def read_2020_labels(text):
    """eval_data_list.csv: a line with a machine type, then its clips as 'file name, original name, label'."""
    lab, typ = {}, None
    for row in csv.reader(io.StringIO(text)):
        if len(row) == 1 and row[0].strip():
            typ = row[0].strip(); lab[typ] = {}
        elif len(row) == 3 and typ:
            lab[typ][row[0]] = int(row[2])
    return lab


def labels_2020():
    b = fetch("https://zenodo.org/records/3951620/files/eval_data_list.csv?download=1", "eval_data_list.csv")
    if b is None:
        return None
    if hashlib.md5(b).hexdigest() != "f16aaa185df9906e588864aaf7cb58ea":
        print("   eval_data_list.csv is not the file our runs used: not used"); return None
    return read_2020_labels(b.decode())


def labels_from_evaluator(url, name, parts, sha):
    """The label and domain files of an official evaluator's repository at one commit: (type, section) → file → (label, domain)."""
    b = fetch(url, name)
    if b is None:
        return None
    with zipfile.ZipFile(io.BytesIO(b)) as zf:
        names = {n.split("/", 1)[1]: n for n in zf.namelist() if "/" in n and not n.endswith("/")}
        want = sorted(n for n in names if n.startswith(tuple(p + "/" for p in parts)) and n.endswith(".csv"))
        blobs = {n: zf.read(names[n]) for n in want}
    h = hashlib.sha256()
    for n in want:                                       # the same fingerprint our training notebooks checked
        h.update(n.encode() + b"\0" + blobs[n])
    if h.hexdigest() != sha:
        print(f"   {name}: the label files are not the ones our runs used: not used"); return None
    rows = lambda n: [r for r in csv.reader(io.StringIO(blobs[n].decode())) if r]
    gt = {}
    for n in want:
        m = re.match(r"ground_truth_data/ground_truth_(\w+?)_(section_\d+)_test\.csv$", n)
        if m:
            dom = {r[0]: ("target" if int(float(r[1])) == 1 else "source") for r in rows(n.replace("ground_truth_data/", "ground_truth_domain/"))}
            gt[m.groups()] = {r[0]: (int(float(r[1])), dom[r[0]]) for r in rows(n)}
    return gt


LABELS = {
    2020: labels_2020(),
    2022: labels_from_evaluator("https://github.com/Kota-Dohi/dcase2022_evaluator/archive/de8719b7d35280ff3a39487c9d1003413eef7818.zip",
                                "dcase2022_evaluator.zip", ("ground_truth_data", "ground_truth_domain"),
                                "d61ad20294be2e127d53ec2bdc1e9f3d1fb4e5cb932f97137ba7e231f2c1f35e"),
    2025: labels_from_evaluator("https://github.com/nttcslab/dcase2025_task2_evaluator/archive/f903124a40a4cdb3faf2b2d6fa69c6a9f1794192.zip",
                                "dcase2025_task2_evaluator.zip", ("ground_truth_data", "ground_truth_domain", "ground_truth_attributes"),
                                "e54a474722f1fb5d878bed257c75444251eeefae4e4dd40a0f5363f8cec68850"),
}
for year, lab in LABELS.items():
    print(year, "labels:", "ready, fingerprint checked" if lab else "NOT available → that test shows the recorded numbers")

## 2 · The measure

**AUC.** Take one faulty clip and one healthy clip at random. AUC is the chance that the faulty one gets the higher score.
50 % is a coin, 100 % is perfect.

**pAUC.** A fire alarm that rings every day gets switched off. pAUC is the same grade, but only in the strict zone: when at most
1 healthy clip in 10 is allowed to raise a false alarm.

**Official score** (tests 2 and 3). Three grades per machine: the AUC on the training condition, the AUC on the new condition, and the
pAUC. The official score is their harmonic mean over all machines, so one weak grade pulls it down.

In [ ]:
EPS = sys.float_info.epsilon
hm = lambda v: len(v) / sum(1.0 / max(float(x), EPS) for x in v)            # harmonic mean
pct = lambda v: round(100 * float(v), 2)


def three(y, s, dom):
    """The three grades of one machine: AUC on the training condition, AUC on the new condition, pAUC.
    Each AUC uses the healthy clips of its own condition and all the faulty clips."""
    y, s, dom = np.asarray(y), np.asarray(s, float), np.asarray(dom)
    faulty = y == 1
    return [roc_auc_score(y[(dom == d) | faulty], s[(dom == d) | faulty]) for d in ("source", "target")] + [roc_auc_score(y, s, max_fpr=0.1)]


def check(what, value, recorded, tol=5e-5):
    ok = abs(float(value) - float(recorded)) <= tol
    print(("   ✔ " if ok else "   ✘ DIFFERS · ") + f"{what}: recomputed {float(value):.4f}, recorded {float(recorded):.4f}")
    return ok


def read_scores(path):
    return {r[0]: float(r[1]) for r in csv.reader(open(path, newline="")) if r}

## 3 · Test 1 · known machines (DCASE 2020 Task 2)

The test clips come from the machines and the conditions the model trained on. Six machine types; the grade of a type is the average
AUC of its machines.

Our system here is **the fusion**: two models listen to each clip and their scores are added.
- **The expert:** a pre-trained audio model (BEATs), fine-tuned on one exercise: *which machine is this?*
- **The trainee:** a small network trained from scratch on the machine's own clips.

In [ ]:
T1 = DATA / "test1_dcase2020"
NAMES = {"fan": "Fan", "pump": "Pump", "slider": "Slide rail", "valve": "Valve", "ToyCar": "Toy car", "ToyConveyor": "Toy conveyor"}
recorded = pd.read_csv(T1 / "recorded_official_evaluator_result.csv", dtype={"machine_id": str}).set_index(["machine_type", "machine_id"])

per_machine = []
for f in sorted((T1 / "fusion").glob("anomaly_score_*.csv")):
    typ, mid = re.match(r"anomaly_score_(\w+)_id_(\d+)\.csv$", f.name).groups()
    sc = read_scores(f)
    lab = (LABELS[2020] or {}).get(typ, {})
    if all(n in lab for n in sc):                        # the labels of this machine are here: recompute
        y, s = [lab[n] for n in sc], list(sc.values())
        auc, pauc, how = roc_auc_score(y, s), roc_auc_score(y, s, max_fpr=0.1), "recomputed"
    else:                                                # no labels: the value our run recorded with the official evaluator
        auc, pauc, how = recorded.loc[(typ, mid), "AUC"], recorded.loc[(typ, mid), "pAUC"], "recorded"
    per_machine.append({"type": typ, "machine": mid, "clips": len(sc), "AUC": auc, "pAUC": pauc, "how": how})
per_machine = pd.DataFrame(per_machine)

redone = per_machine[per_machine.how == "recomputed"]
if len(redone):
    d = max(abs(r.AUC - recorded.loc[(r.type, r.machine), "AUC"]) for r in redone.itertuples())
    print(f"{len(redone)} of {len(per_machine)} machines recomputed from the labels; largest difference to the recorded AUC: {d:.1e}")
else:
    print("no 2020 labels: the table shows the recorded values")

subs = pd.read_csv(T1 / "challenge_submissions_eval_auc.csv")
subs = subs[~subs["system"].str.startswith("DCASE2020_baseline")]
types = list(NAMES)
t1 = pd.DataFrame({"Our system · AUC %": [pct(per_machine[per_machine.type == t].AUC.mean()) for t in types],
                   "Best submission for that type · AUC %": [round(subs[t].max(), 2) for t in types]}, index=[NAMES[t] for t in types])
mean_ours = round(t1["Our system · AUC %"].mean(), 2)
best_mean = round(subs[types].mean(axis=1).max(), 2)
if len(redone) == len(per_machine):
    check("mean AUC of the six types", mean_ours / 100, SUMMARY["test1"]["mean_ours"] / 100)
display(t1)
print(f"Mean of the six types: ours {mean_ours:.2f} %   ·   the best single submission of the {len(subs)}: {best_mean:.2f} %")
print(f"Highest AUC on {(t1.iloc[:, 0] > t1.iloc[:, 1]).sum()} of the 6 types.")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
for i, t in enumerate(reversed(t1.index)):
    a, b = t1.loc[t].iloc[0], t1.loc[t].iloc[1]
    ax.plot([a, b], [i, i], color="#c3c2b7", lw=2, zorder=1)
    ax.scatter(b, i, s=90, color=ORANGE, zorder=2, label="best submission for that type" if i == 0 else None)
    ax.scatter(a, i, s=110, color=BLUE, zorder=3, label="our system" if i == 0 else None)
    lo, hi = min(a, b), max(a, b)
    ax.annotate(f"{lo:.2f}", (lo, i), xytext=(-9, 0), textcoords="offset points", ha="right", va="center")
    ax.annotate(f"{hi:.2f}", (hi, i), xytext=(9, 0), textcoords="offset points", ha="left", va="center")
ax.set_yticks(range(len(t1))); ax.set_yticklabels(list(reversed(t1.index)))
ax.set_xlim(86.5, 101.5); ax.set_xlabel("AUC, %"); ax.grid(axis="y", visible=False)
ax.set_title("Test 1 by machine type", loc="left", fontweight="bold"); ax.legend(loc="upper left", frameon=False)
plt.show()

**What to read here.** The toy conveyor is the weak type. The orange points are the best of all submissions *for that type*, not one
system. The challenge itself ranked by the average rank over AUC and pAUC; here the comparison is by mean AUC.

## 4 · Test 2 · new operating conditions (DCASE 2022 Task 2)

The same machines, at a different speed, load or background noise. There are 990 healthy clips of the training condition and only
10 of the new one, and the test does not say which condition a clip comes from.

**One more idea, ours.** The fusion is not the better listener for every machine type. So, per machine type, the development set
chooses who listens: the expert alone, the trainee alone, or the fusion. The scores themselves are not touched.

In [ ]:
T2 = DATA / "test2_dcase2022"
TYPES22 = ["ToyCar", "ToyTrain", "bearing", "fan", "gearbox", "slider", "valve"]
CAND = {"expert": ("frozen", "expert"), "trainee": ("view_paper", "trainee"), "fusion": ("ZP", "fusion")}   # column in the development files, evaluation folder

# 1 · the choice, on the development set (its labels are in our score files: they come from the clips' file names)
dev = {}
for t in TYPES22:
    for q in ("section_00", "section_01", "section_02"):
        d = pd.read_csv(T2 / "development_scores" / f"scores_{t}_{q}.csv")
        for c, (col, _) in CAND.items():
            dev.setdefault((t, c), []).extend(three(d["label"], d[col], d["domain"]))
dev_type = {t: {c: hm(dev[(t, c)]) for c in CAND} for t in TYPES22}
choice = {t: max(CAND, key=lambda c: (round(dev_type[t][c], 12), c == "fusion")) for t in TYPES22}
display(pd.DataFrame({t: {**{c: pct(dev_type[t][c]) for c in CAND}, "chosen": choice[t]} for t in TYPES22}).T.rename_axis("development set, score %"))

# 2 · the evaluation set
REC22 = json.loads((T2 / "recorded" / "per_type_choice_2022_result.json").read_text())
assert choice == REC22["choice"], "the choice differs from the recorded one"
if LABELS[2022]:
    ev = {}
    for t in TYPES22:
        for c, (_, folder) in CAND.items():
            for q in ("section_03", "section_04", "section_05"):
                sc = read_scores(T2 / "evaluation" / folder / f"anomaly_score_{t}_{q}_test.csv"); gt = LABELS[2022][(t, q)]
                assert sorted(sc) == sorted(gt)
                ev.setdefault((t, c), []).extend(three([gt[n][0] for n in sc], list(sc.values()), [gt[n][1] for n in sc]))
    omega = lambda pick: hm([v for t in TYPES22 for v in ev[(t, pick(t))]])
    r22 = {"Fusion everywhere (the system as it is)": omega(lambda t: "fusion"), "Expert alone": omega(lambda t: "expert"),
           "Trainee alone": omega(lambda t: "trainee"), "Per machine type: the development set's choice": omega(lambda t: choice[t])}
    per_type22 = {t: {"expert": hm(ev[(t, "expert")]), "fusion": hm(ev[(t, "fusion")])} for t in TYPES22}
    check("the system as it is", r22["Fusion everywhere (the system as it is)"], REC22["evaluation"]["as_it_is_fusion_everywhere"])
    check("with the choice per machine type", r22["Per machine type: the development set's choice"], REC22["evaluation"]["with_the_choice"])
else:
    print("no 2022 labels: the recorded numbers")
    S2 = SUMMARY["test2"]
    r22 = {"Fusion everywhere (the system as it is)": S2["fusion_as_it_is"] / 100, "Expert alone": S2["expert_alone"] / 100,
           "Trainee alone": S2["trainee_alone"] / 100, "Per machine type: the development set's choice": S2["per_type_choice"] / 100}
    per_type22 = S2["per_type"]
t2 = pd.DataFrame({"Official score %": {k: pct(v) for k, v in r22.items()}})
t2.loc["Best of the challenge's 31 teams"] = SUMMARY["test2"]["best_team"]
display(t2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(TYPES22))
e = [100 * per_type22[t]["expert"] for t in TYPES22]; f = [100 * per_type22[t]["fusion"] for t in TYPES22]
ax.bar(x - 0.2, e, 0.38, color=ORANGE, label="expert alone"); ax.bar(x + 0.2, f, 0.38, color=BLUE, label="fusion")
for i, t in enumerate(TYPES22):
    if choice[t] in ("expert", "fusion"):               # mark the listener the development set chose
        ax.annotate("chosen", (i - 0.2, e[i]) if choice[t] == "expert" else (i + 0.2, f[i]), xytext=(0, 4), textcoords="offset points",
                    ha="center", fontsize=8.5, color="#52514e")
ax.set_xticks(x); ax.set_xticklabels(TYPES22); ax.set_ylabel("score on the evaluation set, %"); ax.set_ylim(0, 105); ax.grid(axis="x", visible=False)
ax.set_title("Test 2 by machine type: who listens better?", loc="left", fontweight="bold")
ax.legend(frameon=False, ncol=2, loc="lower right", bbox_to_anchor=(1.0, 1.0))
plt.show()

print("Six other ideas, tried before this one (change of the official score on the development set, in points). None was kept:")
display(pd.DataFrame(SUMMARY["test2"]["development"]["ideas"]).rename(columns={"n": "#", "what": "idea", "change_points": "change"}).set_index("#"))

**What to read here.** The development set gives the toy car, the gearbox and the valve to the expert, and the rest to the fusion.
On the evaluation set the expert is ahead on the toy car and the gearbox. The six other ideas moved the score from one condition to
the other and added nothing.

## 5 · Test 3 · new machine types (DCASE 2025 Task 2)

Eight machine types that share nothing with the machines used for development. Each comes with healthy clips only, and there is
nothing to tune on. Four steps:

1. **The system as it is:** expert + trainee.
2. **The trainee taken out.**
3. **The expert's exercise changed:** not only *which machine is this?* but *which machine, and at which setting?* (speed, load…).
4. **Two pre-trained models added, without training.** Three models, each summarising a clip in 11 ways: 33 listeners, equal votes.

In [ ]:
T3 = DATA / "test3_dcase2025"; Q = "section_00"
J35 = json.loads((T3 / "listeners_bank" / "pooling_first_shot.json").read_text())
TE, POOLS = J35["types_evaluation"], J35["poolings"]
REC35 = json.loads((T3 / "recorded" / "analysis_n35.json").read_text())

if LABELS[2025]:
    def omega25(score_of):
        v = []
        for t in TE:
            gt = LABELS[2025][(t, Q)]; names, s = score_of(t)
            assert sorted(names) == sorted(gt), t
            v.extend(three([gt[n][0] for n in names], s, [gt[n][1] for n in names]))
        return hm(v)

    def from_folder(folder):
        def score_of(t):
            sc = read_scores(T3 / "evaluation" / folder / f"anomaly_score_{t}_{Q}_test.csv")
            return list(sc), list(sc.values())
        return score_of

    # step 4 · the 33 listeners. Each one's scores are first put on a common scale, using only healthy reference clips:
    # subtract the median of its 'leave-one-out' reference scores and divide by their robust spread. Then the 33 are averaged.
    BANK = {t: np.load(T3 / "listeners_bank" / f"bank_{t}.npz", allow_pickle=False) for t in TE}
    KEYS = BANK[TE[0]]["keys"].tolist(); IDX = {k: i for i, k in enumerate(KEYS)}
    C33 = [f"AT|F|{p}" for p in POOLS] + [f"FZ|B3|{p}|Ev" for p in POOLS] + [f"FZ|B|{p}|Ev" for p in POOLS]
    is_ratio = lambda k: k.startswith("AT|") or k.endswith("|C1")           # these scores are ratios: they enter as logarithms
    raw = lambda t, k, part: (np.log(np.maximum(BANK[t][part][IDX[k]].astype(np.float64), 1e-12)) if is_ratio(k) else BANK[t][part][IDX[k]].astype(np.float64))

    def z(t, k):
        ref = raw(t, k, "loo"); med = np.median(ref); spread = 1.4826 * np.median(np.abs(ref - med))
        return (raw(t, k, "test") - med) / max(spread, 1e-9)

    votes = lambda t: (BANK[t]["test_names"].tolist(), np.stack([z(t, k) for k in C33]).mean(0))
    steps = [omega25(from_folder("fusion")), omega25(from_folder("expert")), omega25(from_folder("expert_machine_settings")), omega25(votes)]
    for nm, v, r in zip(("step 1", "step 2", "step 3"), steps, [s["score"] / 100 for s in SUMMARY["test3"]["steps"]]):
        check(nm, v, r, tol=6e-5)
    check("step 4", steps[3], REC35["readings"]["A5"]["evaluation"]["omega"], tol=1e-9)
else:
    print("no 2025 labels: the recorded numbers")
    steps = [s["score"] / 100 for s in SUMMARY["test3"]["steps"]]

t3 = pd.DataFrame({"Official score %": [pct(v) for v in steps]},
                  index=["1 · expert + trainee (the system as it is)", "2 · expert alone", "3 · expert, taught the machine settings",
                         "4 · + two pre-trained models, 33 listeners, equal votes"])
t3["step"] = ["", *[f"+{pct(b) - pct(a):.1f}" for a, b in zip(steps, steps[1:])]]
t3.loc["Best of the challenge's 35 teams"] = [SUMMARY["test3"]["best_team"], ""]
display(t3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ys = [pct(v) for v in steps]; best = SUMMARY["test3"]["best_team"]
ax.axhline(best, color=ORANGE, lw=2); ax.annotate(f"best challenge team {best:.2f}", (-0.4, best), xytext=(0, 5), textcoords="offset points")
ax.plot(range(4), ys, color=BLUE, lw=2); ax.scatter(range(4), ys, s=110, color=BLUE, zorder=3)
for i, v in enumerate(ys):
    ax.annotate(f"{v:.2f}", (i, v), xytext=(0, 11 if v > best else -17), textcoords="offset points", ha="center", fontweight="bold")
ax.set_xticks(range(4)); ax.set_xticklabels(["expert + trainee", "expert alone", "expert, taught the\nmachine settings", "+ two pre-trained\nmodels, equal votes"])
ax.set_xlim(-0.45, 3.45); ax.set_ylim(48, 67); ax.set_ylabel("official score, %"); ax.grid(axis="x", visible=False)
ax.set_title("Test 3 in four steps (the axis starts at 48 %)", loc="left", fontweight="bold")
plt.show()

nk = SUMMARY["test3"]["not_kept"]
print("What did not help (recorded):")
print(f"   steps 2 and 3 added together: {nk['two_experts_together']:.2f} %")
print(f"   a fusion that weights its own listeners: {nk['self_weighted_fusion']:.2f} %")
print(f"   a published training-free recipe on our files: {nk['published_training_free_recipe_on_our_files']:.2f} % (its paper reports 62.4 %)")
print(f"   the target was not reached: {SUMMARY['test3']['published']['arXiv_2609.27362']} % is published (arXiv 2609.27362)")

**What to read here.** Taking the trainee out is the largest step: on machine types it has never met, the small network trained from
scratch pulls the score down. The last step is small.

## 6 · Factory noise, one microphone or eight

MIMII was recorded with a ring of eight microphones, at three noise levels: the machine louder than the factory (6 dB), as loud
(0 dB), and the factory louder than the machine (−6 dB). These numbers are **recorded** by our training runs (average AUC over the
four pumps); this notebook does not recompute them.

In [ ]:
N = SUMMARY["noise"]; levels = ["6dB", "0dB", "-6dB"]
series = [("Family 5 (the expert) · eight microphones", [N["family5_fine_tuned"]["ARR"][k]["pump"] for k in levels], BLUE, "-"),
          ("Family 5 (the expert) · one microphone", [N["family5_fine_tuned"]["MIC"][k]["pump"] for k in levels], BLUE, "--"),
          ("Family 2 (normal cloud) · eight microphones", [N["families_1_to_4_pumps"]["F2"][k]["BF"] for k in levels], ORANGE, "-"),
          ("Family 2 (normal cloud) · one microphone", [N["families_1_to_4_pumps"]["F2"][k]["CH0"] for k in levels], ORANGE, "--")]
fig, ax = plt.subplots(figsize=(9, 4))
for name, ys, col, ls in series:
    ax.plot(range(3), ys, color=col, ls=ls, lw=2, marker="o", label=name)
ax.set_xticks(range(3)); ax.set_xticklabels(["6 dB\nmachine louder", "0 dB\nequal", "−6 dB\nfactory louder"]); ax.set_ylim(0.70, 1.01)
ax.set_ylabel("average AUC, four pumps"); ax.grid(axis="x", visible=False)
ax.set_title("The pumps in factory noise", loc="left", fontweight="bold"); ax.legend(frameon=False, loc="lower left", fontsize=9.5)
plt.show()
display(pd.DataFrame({name: [round(v, 3) for v in ys] for name, ys, *_ in series}, index=["6 dB", "0 dB", "−6 dB"]).T)

**What to read here.** When the factory is louder than the pump, eight microphones bring back most of what one microphone loses.
One microphone remains the practical choice in a plant: simple, low-cost, realistic. The array is what to reach for when the noise wins.

## 7 · Five families of methods, on the same tests

From the simplest method to the newest (recorded by our runs). Family 5's row holds its best version in each column.

In [ ]:
F = SUMMARY["families"]
fam = pd.DataFrame({"Four pumps, 6 dB · AUC": {k: round(F["four_pumps_6dB_one_microphone_auc"][k], 2) for k in F["names"]},
                    "Test 1 · mean AUC %": F["test1_mean_auc"], "Test 2 · official score %": F["test2_official"], "Test 3 · official score %": F["test3_official"]})
fam.index = [f"{i} · {F['names'][k]}" for i, k in enumerate(F["names"], 1)]
display(fam)

**What to read here.** On the easy case (a pump louder than the factory) a method from the 1990s is nearly enough. The gap opens when
the test gets harder.

## Limits

- **Not challenge entries.** The challenges are closed, the labels are public, and the system uses pre-trained models that the 2020
  and 2022 teams did not have.
- **Test 1** is compared by mean AUC, while the challenge ranked by the average rank over AUC and pAUC. The table of the 116
  submissions was transcribed from the challenge's results page; only five of its rows were checked against a second source.
- **Test 2:** the choice per machine type used development clips only, but it was made after the evaluation result of the system as
  it is (70.63 %) had been seen. Its 0.53 points over the first team are inside the noise of this test.
- **Test 3:** the last step (0.99 points) is under the 1-point bar our own rules use for a clear difference, and it was chosen on an
  evaluation set that had been read before. On this test the development set does not predict the evaluation set.
- **Recordings, not a plant.** Clips of 10 seconds from laboratory and toy machines, with faults made on purpose and factory noise
  mixed in. This is a teaching project, not a machine-protection system.

## Sources

- **Data:** MIMII (Purohit et al., 2019, Zenodo 3384388); DCASE 2020 Task 2 (Zenodo 3678171, 3727685, 3841772, 3951620); DCASE 2022
  Task 2 (Zenodo 6355122, 6462969, 6586456); DCASE 2025 Task 2 (Zenodo 15097779, 15392814, 15519362).
- **Models:** BEATs (Chen et al., 2022); ECHO; TASTgram-MFN with Noisy-ArcMix (Choi and Choi, ICASSP 2024).
- **Official evaluators:** `y-kawagu/dcase2020_task2_evaluator`, `Kota-Dohi/dcase2022_evaluator`, `nttcslab/dcase2025_task2_evaluator`.
- **The project:** https://github.com/EngMtaleb/chemai/tree/main/projects/p04_machine_sound

*Implementation assisted by AI tools. The engineering framing, interpretation and validation are the author's.*